# Retrieve BindingDB data

The purpose of this notebook is to retrieve affinity data based on local crystal structures from Binding DB and the PDB.

In [3]:
import requests
import time
import csv

import pathlib
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.notebook import tqdm

Protein and data variables.

In [4]:
readout = 'affinity'

In [5]:
# define paths
HERE = Path(pathlib.Path.cwd())
DATA = HERE / f"data_{readout}"
DATA.mkdir(parents=True, exist_ok=True)

### Download bioactivity data from BindingDB and PDB.

In [6]:
root_dir = Path(DATA)

records = []
for uniprot_dir in root_dir.iterdir():
    if uniprot_dir.is_dir():
        uniprot_id = uniprot_dir.name
        for pdb_file in uniprot_dir.glob("*.pdb"):
            pdb_id = pdb_file.stem.upper()
            records.append({"uniprot_id": uniprot_id, "pdb_id": pdb_id, "pdb_path": str(pdb_file)})

df_pdbs = pd.DataFrame(records)
df_pdbs.head()


,uniprot_id,pdb_id,pdb_path
0,O08675,2PUX,/home/corey/Documents/comp_chem/ml/affinity/si...
1,Q99788,7YKD,/home/corey/Documents/comp_chem/ml/affinity/si...
2,Q99788,9L3Z,/home/corey/Documents/comp_chem/ml/affinity/si...
3,Q99788,9L3W,/home/corey/Documents/comp_chem/ml/affinity/si...
4,Q99788,8ZJG,/home/corey/Documents/comp_chem/ml/affinity/si...


In [7]:
import asyncio
import aiohttp
import json
import time
import pandas as pd
from tqdm.asyncio import tqdm_asyncio

# --------------------------
# Config
# --------------------------
CONCURRENCY = 4          # polite concurrency
RETRIES = 4
BACKOFF = 2.0
MAX_BACKOFF = 30
TIMEOUT = 90
SLEEP_BETWEEN_BATCHES = 2.5

CACHE_DIR = Path("bindingdb_cache")
CACHE_DIR.mkdir(exist_ok=True)

def cache_path(kind, query_id):
    """Generate filename for a cached UniProt or PDB query."""
    return CACHE_DIR / f"{kind}_{query_id}.json"

def load_from_cache(kind, query_id):
    """Return cached data if available, else None."""
    path = cache_path(kind, query_id)
    if path.exists():
        try:
            with open(path) as f:
                return json.load(f)
        except Exception:
            return None
    return None

def save_to_cache(kind, query_id, data):
    """Write successful query results to cache."""
    if not data:
        return
    path = cache_path(kind, query_id)
    try:
        with open(path, "w") as f:
            json.dump(data, f)
    except Exception as e:
        print(f"Cache write failed for {query_id}: {e}")

# --------------------------
# Helper async functions
# --------------------------
async def fetch_with_retries(session, url, retries=RETRIES, backoff=BACKOFF):
    """Fetch a URL with retry and exponential backoff."""
    for attempt in range(retries):
        try:
            async with session.get(url, timeout=TIMEOUT) as resp:
                txt = await resp.text()
                if resp.status == 200 and txt.strip():
                    return txt
                elif resp.status in (429, 503):
                    await asyncio.sleep(min(backoff * (2 ** attempt), MAX_BACKOFF))
                else:
                    return None
        except Exception:
            await asyncio.sleep(min(backoff * (2 ** attempt), MAX_BACKOFF))
    return None


async def fetch_bindingdb_uniprot(session, uniprot_id):
    # Try cache first
    cached = load_from_cache("uniprot", uniprot_id)
    if cached is not None:
        return uniprot_id, cached

    # Fetch from BindingDB if not cached
    url = f"https://bindingdb.org/rest/getLigandsByUniprot?uniprot={uniprot_id};10000&response=application/json"
    text = await fetch_with_retries(session, url)
    if not text:
        return uniprot_id, None
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            data = [data]
        save_to_cache("uniprot", uniprot_id, data)   # save to cache
        return uniprot_id, data
    except json.JSONDecodeError:
        return uniprot_id, None


async def fetch_bindingdb_pdb(session, pdb_id):
    cached = load_from_cache("pdb", pdb_id)
    if cached is not None:
        return pdb_id, cached

    url = f"https://bindingdb.org/rest/getLigandsByPDBs?pdb={pdb_id}&response=application/json"
    text = await fetch_with_retries(session, url)
    if not text:
        return pdb_id, None
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            data = [data]
        save_to_cache("pdb", pdb_id, data)
        return pdb_id, data
    except json.JSONDecodeError:
        return pdb_id, None


# --------------------------
# Orchestration
# --------------------------
from tqdm.asyncio import tqdm_asyncio
import asyncio

from tqdm.notebook import tqdm
import asyncio
import time

async def query_bindingdb_from_df(df_pdbs, cpus=4, log_every=20):
    """
    Query BindingDB by UniProt ID, with fallback to PDB IDs.
    Notebook-safe, cache-aware, with live progress and periodic summaries.
    """
    sem = asyncio.Semaphore(cpus)
    timeout = aiohttp.ClientTimeout(total=None)
    results = []

    async with aiohttp.ClientSession(timeout=timeout) as session:
        uniprot_ids = df_pdbs["uniprot_id"].unique().tolist()
        total_ids = len(uniprot_ids)
        print(f"Querying {total_ids} UniProt IDs...")

        async def uni_worker(uid):
            async with sem:
                cached = load_from_cache("uniprot", uid)
                if cached is not None:
                    await asyncio.sleep(0)
                    return uid, cached, True
                data = await fetch_bindingdb_uniprot(session, uid)
                return data[0], data[1], False 

        tasks = [asyncio.create_task(uni_worker(uid)) for uid in uniprot_ids]
        uni_results, n_cached, n_found, n_missing = [], 0, 0, 0

        pbar = tqdm(total=total_ids, desc="UniProt queries", unit="ID")
        start_time = time.time()

        for i, task in enumerate(asyncio.as_completed(tasks), start=1):
            uid, entries, was_cached = await task
            uni_results.append((uid, entries))
            pbar.update(1)

            if was_cached:
                n_cached += 1
            if entries:
                n_found += 1
            else:
                n_missing += 1

            # log progress every N items
            if i % log_every == 0 or i == total_ids:
                elapsed = time.time() - start_time
                rate = i / elapsed if elapsed > 0 else 0
                print(f"[UniProt] {i}/{total_ids} processed "
                      f"({n_cached} cached, {n_found} found, {n_missing} missing) "
                      f"— {rate:.2f} IDs/s")

        pbar.close()
        print(f"UniProt stage complete: {n_found} with data, {n_missing} with no data, {n_cached} from cache")

        #  Fallback to PDB.
        pdb_results = []  # define it early to ensure it exists

        missing_uniprots = [uid for uid, entries in uni_results if not entries]
        if missing_uniprots:
            pdbs_to_query = df_pdbs[df_pdbs["uniprot_id"].isin(missing_uniprots)]["pdb_id"].unique().tolist()
            print(f"Trying fallback PDB queries for {len(pdbs_to_query)} PDBs...")

            async def pdb_worker(pid):
                async with sem:
                    cached = load_from_cache("pdb", pid)
                    if cached is not None:
                        await asyncio.sleep(0)
                        return pid, cached, True
                    data = await fetch_bindingdb_pdb(session, pid)
                    return data[0], data[1], False

            pdb_results, n_cached_pdb, n_found_pdb, n_missing_pdb = [], 0, 0, 0
            tasks = [asyncio.create_task(pdb_worker(pid)) for pid in pdbs_to_query]

            pbar = tqdm(total=len(tasks), desc="PDB queries", unit="PDB")
            start_time = time.time()

            for j, task in enumerate(asyncio.as_completed(tasks), start=1):
                pid, entries, was_cached = await task
                pdb_results.append((pid, entries))
                pbar.update(1)

                if was_cached:
                    n_cached_pdb += 1
                if entries:
                    n_found_pdb += 1
                else:
                    n_missing_pdb += 1

                if j % log_every == 0 or j == len(tasks):
                    elapsed = time.time() - start_time
                    rate = j / elapsed if elapsed > 0 else 0
                    print(f"[PDB] {j}/{len(tasks)} processed "
                        f"({n_cached_pdb} cached, {n_found_pdb} found, {n_missing_pdb} missing) "
                        f"— {rate:.2f} PDBs/s")

            pbar.close()
            print(f"PDB stage complete: {n_found_pdb} with data, {n_missing_pdb} missing, {n_cached_pdb} from cache")


        # --- Combine UniProt + PDB results ---
        all_rows = []

        # Add UniProt rows
        for uid, entries in uni_results:
            if entries:
                for e in entries:
                    e["query_type"] = "uniprot"
                    e["query_id"] = uid
                    all_rows.append(e)

        # Build mapping PDB → UniProt
        pdb_to_uniprot = dict(zip(df_pdbs["pdb_id"], df_pdbs["uniprot_id"]))

        # PDB results (now correctly annotated)
        for pid, entries in pdb_results:
            if entries:
                uni_id = pdb_to_uniprot.get(pid)
                for e in entries:
                    e["query_type"] = "pdb"
                    e["query_id"] = uni_id or pid  # fallback
                    # e["pdb_id"] = pid
                    all_rows.append(e)

        print(f"Total collected entries: {len(all_rows)}")

        return pd.DataFrame(all_rows)


In [8]:
bindingdb_df = await query_bindingdb_from_df(df_pdbs, cpus=16)
bindingdb_df.head()

Querying 201 UniProt IDs...


UniProt queries:   0%|          | 0/201 [00:00<?, ?ID/s]

[UniProt] 20/201 processed (20 cached, 20 found, 0 missing) — 904.01 IDs/s
[UniProt] 40/201 processed (40 cached, 40 found, 0 missing) — 1372.44 IDs/s
[UniProt] 60/201 processed (57 cached, 57 found, 3 missing) — 94.01 IDs/s
[UniProt] 80/201 processed (73 cached, 73 found, 7 missing) — 123.28 IDs/s
[UniProt] 100/201 processed (91 cached, 91 found, 9 missing) — 151.44 IDs/s
[UniProt] 120/201 processed (107 cached, 107 found, 13 missing) — 140.46 IDs/s
[UniProt] 140/201 processed (117 cached, 119 found, 21 missing) — 0.36 IDs/s
[UniProt] 160/201 processed (129 cached, 131 found, 29 missing) — 0.41 IDs/s
[UniProt] 180/201 processed (149 cached, 151 found, 29 missing) — 0.46 IDs/s
[UniProt] 200/201 processed (152 cached, 154 found, 46 missing) — 0.25 IDs/s
[UniProt] 201/201 processed (152 cached, 154 found, 47 missing) — 0.26 IDs/s
UniProt stage complete: 154 with data, 47 with no data, 152 from cache
Trying fallback PDB queries for 269 PDBs...


PDB queries:   0%|          | 0/269 [00:00<?, ?PDB/s]

[PDB] 20/269 processed (9 cached, 9 found, 11 missing) — 9.16 PDBs/s
[PDB] 40/269 processed (19 cached, 19 found, 21 missing) — 10.88 PDBs/s
[PDB] 60/269 processed (26 cached, 26 found, 34 missing) — 9.68 PDBs/s
[PDB] 80/269 processed (34 cached, 34 found, 46 missing) — 10.61 PDBs/s
[PDB] 100/269 processed (40 cached, 40 found, 60 missing) — 11.05 PDBs/s
[PDB] 120/269 processed (43 cached, 43 found, 77 missing) — 10.32 PDBs/s
[PDB] 140/269 processed (52 cached, 52 found, 88 missing) — 10.83 PDBs/s
[PDB] 160/269 processed (68 cached, 68 found, 92 missing) — 11.42 PDBs/s
[PDB] 180/269 processed (79 cached, 79 found, 101 missing) — 11.68 PDBs/s
[PDB] 200/269 processed (84 cached, 84 found, 116 missing) — 11.31 PDBs/s
[PDB] 220/269 processed (104 cached, 104 found, 116 missing) — 12.41 PDBs/s
[PDB] 240/269 processed (110 cached, 110 found, 130 missing) — 12.38 PDBs/s
[PDB] 260/269 processed (119 cached, 119 found, 141 missing) — 12.17 PDBs/s
[PDB] 269/269 processed (119 cached, 119 found, 

,getLindsByUniprotResponse,query_type,query_id,getLindsByPDBsResponse
0,"{'bdb.hit': '111', 'bdb.length': 'NA', 'bdb.un...",uniprot,Q99788,NaN
1,"{'bdb.hit': '826', 'bdb.length': 'NA', 'bdb.un...",uniprot,P32246,NaN
2,"{'bdb.hit': '113', 'bdb.length': 'NA', 'bdb.un...",uniprot,Q96LB2,NaN
3,"{'bdb.hit': '186', 'bdb.length': 'NA', 'bdb.un...",uniprot,P51686,NaN
4,"{'bdb.hit': '59', 'bdb.length': 'NA', 'bdb.uni...",uniprot,P22888,NaN


In [9]:
bindingdb_df.tail()

,getLindsByUniprotResponse,query_type,query_id,getLindsByPDBsResponse
268,NaN,pdb,P35372,{'affinities': [{'query': 'Mu-type opioid rece...
269,NaN,pdb,P35372,{'affinities': [{'query': 'Guanine nucleotide-...
270,NaN,pdb,P35372,{'affinities': [{'query': 'Mu-type opioid rece...
271,NaN,pdb,P35372,{'affinities': [{'query': 'Guanine nucleotide-...
272,NaN,pdb,P35372,{'affinities': [{'query': 'Mu-type opioid rece...


Now comes matching and exploding the output from these two data sources. First we need to find the structure to 3-letter code matching for ligands.

In [ ]:
from rdkit import Chem
from rdkit.Chem import PandasTools

# Load the SDF file
sdf_path = 'ccd_tools/components-pub.sdf'
supplier = Chem.SDMolSupplier(sdf_path)

# Create a DataFrame from the SDF file
sdf_df = PandasTools.LoadSDF(sdf_path, smilesName="SMILES", includeFingerprints=False)

# Build dictionary: 3-letter code (usually in column 'ID' or similar) → canonical SMILES
ccd_dict = {}
for idx, row in sdf_df.iterrows():
    chem_comp_id = row['ID']  # Adjust column name if needed
    mol = row['ROMol']         # Default column created by PandasTools.LoadSDF for mol objects
    if mol:
        smiles = Chem.MolToSmiles(mol, canonical=True)
        ccd_dict[chem_comp_id] = smiles

print(f"Loaded {len(ccd_dict)} CCD entries")
print(list(ccd_dict.items())[:5])

In [ ]:
from pathlib import Path
import os
import pandas as pd
import json
from Bio.PDB import PDBParser
# from tqdm import tqdm


# -----------------------------
# 1. Parse Ligands from Local PDBs
# -----------------------------
def extract_ligands_from_pdb(pdb_path):
    parser = PDBParser(QUIET=True)
    try:
        structure = parser.get_structure("x", pdb_path)
    except Exception:
        return []
    ligands = set()
    skip = {"HOH", "WAT", "SO4", "CL", "NA", "K", "MG", "CA"}
    for model in structure:
        for chain in model:
            for residue in chain:
                hetflag, _, _ = residue.get_id()
                if hetflag.strip() and residue.get_resname().strip() not in skip:
                    ligands.add(residue.get_resname().strip())
    return list(ligands)


def build_pdb_ligand_map_follow_symlinks(root_dir):
    """Recursively traverse symlinked directories and collect ligands."""
    pdb_ligands = {}
    root_dir = Path(root_dir)
    for dirpath, dirnames, filenames in tqdm(os.walk(root_dir, followlinks=True), desc="Scanning PDBs"):
        for f in filenames:
            if f.lower().endswith(".pdb"):
                pdb_path = Path(dirpath) / f
                pdb_id = pdb_path.stem.upper()
                ligs = extract_ligands_from_pdb(pdb_path)
                if ligs:
                    pdb_ligands[pdb_id] = ligs
    return pdb_ligands


# -----------------------------
# 2. Merge BindingDB Results
# -----------------------------
def combine_bindingdb_results(bindingdb_df):
    df = bindingdb_df.copy()

    def pick_binding_dict(row):
        for key in row.index:
            if isinstance(row[key], dict) and "affinity" in json.dumps(row[key]):
                return row[key]
        return None

    df["binding_record"] = df.apply(pick_binding_dict, axis=1)
    df = df[df["binding_record"].notnull()].reset_index(drop=True)

    def extract_affinities(record):
        if not record:
            return []
        if "bdb.affinities" in record:
            affs = record["bdb.affinities"]
            return [
                {
                    "smile": a.get("bdb.smile"),
                    "monomerid": a.get("bdb.monomerid"),
                    "affinity": a.get("bdb.affinity"),
                    "affinity_type": a.get("bdb.affinity_type"),
                }
                for a in affs
            ]
        elif "affinities" in record:
            return record["affinities"]
        return []

    df["affinities"] = df["binding_record"].apply(extract_affinities)
    df = df.explode("affinities").reset_index(drop=True)
    aff_df = pd.json_normalize(df["affinities"]).add_prefix("aff_")
    df = pd.concat([df.drop(columns=["affinities"]), aff_df], axis=1)
    df = df.drop(columns=["binding_record"])

    df = df.dropna(subset=["aff_smile", "aff_affinity"])
    df = df.drop_duplicates(
        subset=["aff_monomerid", "aff_affinity", "aff_affinity_type", "aff_smile"]
    ).reset_index(drop=True)
    return df


# -----------------------------
# 3. Match Ligands to PDBs
# -----------------------------
def match_affinities_to_pdbs(bindingdb_df, pdb_ligands, ccd_dict=None):
    matches = []
    # debug_count = 0

    for idx, row in tqdm(bindingdb_df.iterrows(), total=len(bindingdb_df), desc="Matching ligands"):
        smi = row["aff_smile"]
        monomerid = row["aff_monomerid"]
        affinity = row["aff_affinity"]
        affinity_type = row["aff_affinity_type"]
        uniprot = row.get("query_id", None)
        source = row.get("query_type", None)

        # Canonicalize BindingDB SMILES
        bdb_mol = Chem.MolFromSmiles(smi)
        if bdb_mol is None:
            # if debug_count < debug_limit:
            #     print(f"Skipping invalid BindingDB SMILES at index {idx}: {smi}")
            continue
        bdb_canonical = Chem.MolToSmiles(bdb_mol, canonical=True)

        # if debug_count < debug_limit:
        #     print(f"\n--- BindingDB entry {idx} ---")
        #     print(f"Original SMILES: {smi}")
        #     print(f"Canonical SMILES: {bdb_canonical}")
        #     print(f"Monomer ID: {monomerid}")
        #     print(f"Affinity: {affinity} {affinity_type}")
        #     print(f"Source: {source}, UniProt: {uniprot}")

        for pdb_id, lig_list in pdb_ligands.items():
            # Convert 3-letter codes to canonical SMILES
            lig_smiles = []
            for l in lig_list:
                smi = ccd_dict.get(l)
                if smi:
                    lig_smiles.append(smi)

            # if debug_count < debug_limit:
            #     print(f"Checking PDB {pdb_id} ligands (canonical SMILES): {lig_smiles}")

            if bdb_canonical in lig_smiles:
                # if debug_count < debug_limit:
                #     print(f"--> Match found for PDB {pdb_id}")

                matches.append({
                    "pdb_id": pdb_id,
                    "uniprot_id": uniprot,
                    "ligand_smile": bdb_canonical,
                    "monomerid": monomerid,
                    "affinity": affinity,
                    "affinity_type": affinity_type,
                    "source": source,
                })
                # debug_count += 1

            # if debug_count >= debug_limit:
            #     print("\nDebug limit reached, stopping further output.")
            #     break

        # if debug_count >= debug_limit:
        #     break

    matched_df = pd.DataFrame(matches)
    return matched_df


# -----------------------------
# 4. Main pipeline
# -----------------------------
def build_structure_affinity_dataset(pdb_root, bindingdb_df, ccd_dict=None):
    print("🔹 Parsing local PDBs...")
    pdb_ligands = build_pdb_ligand_map_follow_symlinks(pdb_root)

    print("🔹 Combining BindingDB results...")
    bdb_clean = combine_bindingdb_results(bindingdb_df)

    print("🔹 Matching ligands to PDB structures...")
    dataset = match_affinities_to_pdbs(bdb_clean, pdb_ligands, ccd_dict)

    print(f"Final dataset: {len(dataset)} structure–ligand–affinity pairs")
    return dataset


In [14]:
# import os

# pdb_ligands = build_pdb_ligand_map_follow_symlinks(DATA)
# print(f"Found ligands for {len(pdb_ligands)} PDBs")

In [15]:
# Example usage:
final_df = build_structure_affinity_dataset(DATA, bindingdb_df, ccd_dict=ccd_dict)
final_df.head()


🔹 Parsing local PDBs...


Scanning PDBs: 0it [00:00, ?it/s]

🔹 Combining BindingDB results...
🔹 Matching ligands to PDB structures...


Matching ligands:   0%|          | 0/283035 [00:00<?, ?it/s]

[13:15:01] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 16 ignored
[13:15:01] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 16 ignored
[13:15:01] improperly formatted w block
[13:15:01] improperly formatted w block
[13:15:04] atom 114 is not associated with bond 11(10-12) in w block
[13:15:07] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 16 ignored
[13:15:10] improperly formatted w block
[13:15:10] improperly formatted w block
[13:15:17] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 16 ignored
[13:15:20] SMILES Parse Error: syntax error while parsing: null
[13:15:20] SMILES Parse Error: Failed parsing SMILES 'null' for input: 'null'
[13:15:20] SMILES Parse Error: syntax error while parsing: null
[13:15:20] SMILES Parse Error: Failed parsing SMILES 'null' for input: 'null'
[13:15:30] Warning: ambiguous stereochemistry - zero final chiral volume - at atom 16 ignored
[13:15:31] atom 1

✅ Final dataset: 6753 structure–ligand–affinity pairs


,pdb_id,uniprot_id,ligand_smile,monomerid,affinity,affinity_type,source
0,8HJ5,Q96LB2,COc1ccccc1S(=O)(=O)Nc1ccc(C)cc1Oc1ccc2c(N)nccc2c1,50509101,50,EC50,uniprot
1,8DWH,Q96LB2,COc1ccccc1S(=O)(=O)Nc1ccc(C)cc1Oc1ccc2c(N)nccc2c1,50509101,50,EC50,uniprot
2,8DWG,Q96LB2,CCOc1ccccc1NC(=O)c1ccccc1NS(=O)(=O)C1CC1,50605929,124,EC50,uniprot
3,5LWE,P51686,CC(C)(C)c1ccc(S(=O)(=O)Nc2ccc(Cl)cc2C(=O)c2cc[...,50398334,2.6,IC50,uniprot
4,5LWE,P51686,CC(C)(C)c1ccc(S(=O)(=O)Nc2ccc(Cl)cc2C(=O)c2cc[...,50398334,1.1,Ki,uniprot


In [16]:
final_df.to_csv("structure_affinities.csv", index=False)

In [7]:
# Map UniProt → all PDBs
uniprot_to_pdbs = (
    df_pdbs.groupby("uniprot_id")["pdb_id"]
    .apply(list)
    .to_dict()
)

# Map PDB → UniProt
pdb_to_uniprot = dict(zip(df_pdbs["pdb_id"], df_pdbs["uniprot_id"]))

In [8]:
bdb_raw = bindingdb_df.copy()

def pick_binding_dict(row):
    """
    Return the dictionary that contains BindingDB data,
    regardless of whether it's from UniProt or PDB query.
    """
    if isinstance(row.get("getLindsByUniprotResponse"), dict):
        return row["getLindsByUniprotResponse"]
    elif isinstance(row.get("getLindsByPDBsResponse"), dict):
        return row["getLindsByPDBsResponse"]
    else:
        return None

In [ ]:
# Extract the binding data
bdb_raw["binding_record"] = bdb_raw.apply(pick_binding_dict, axis=1)
bdb_records = bdb_raw[bdb_raw["binding_record"].notnull()].reset_index(drop=True)

bdb_raw

print(f"Found {len(bdb_records)} BindingDB or PDB responses.")

def extract_affinities(row):
    rec = row['binding_record']
    if rec is None or pd.isna(rec):
        return []

    if row['query_type'] == 'uniprot':
        # bindingdb style keys
        aff_list = rec.get('bdb.affinities', [])
        affinities = []
        for a in aff_list:
            affinities.append({
                'affinity': a.get('bdb.affinity'),
                'affinity_type': a.get('bdb.affinity_type'),
                'smile': a.get('bdb.smile'),
                'monomerid': a.get('bdb.monomerid'),
                # You can add more fields if needed
            })
        return affinities

    elif row['query_type'] == 'pdb':
        # pdb style keys
        aff_list = rec.get('affinities', [])
        # They are already formatted properly
        return aff_list

    else:
        return []

def assign_pdb_ids(row, uniprot_to_pdbs):
    if row['query_type'] == 'uniprot':
        return uniprot_to_pdbs.get(row['query_id'], [])
    elif row['query_type'] == 'pdb':
        return [row['query_id']]
    else:
        return []

# Extract affinities as lists of dicts
bdb_raw['affinities_extracted'] = bdb_raw.apply(extract_affinities, axis=1)

# Explode the affinities so each row has exactly one affinity dict
bdb_exp = bdb_raw.explode('affinities_extracted').reset_index(drop=True)

# Normalize the dicts in 'affinities_extracted' column into separate columns
affinities_df = pd.json_normalize(bdb_exp['affinities_extracted'])

# Join these columns back to the main dataframe
bdb_exp = bdb_exp.join(affinities_df)

# Drop temporary columns if no longer needed
bdb_exp = bdb_exp.drop(columns=['affinities_extracted', 'binding_record'])

# Make sure you have your uniprot_to_pdbs mapping dictionary ready

bdb_exp['pdb_ids'] = bdb_exp.apply(lambda row: assign_pdb_ids(row, uniprot_to_pdbs), axis=1)

# Explode pdb_ids to get one row per pdb_id + affinity combo
bdb_final = bdb_exp.explode('pdb_ids').reset_index(drop=True)

Found 285 BindingDB or PDB responses.


In [25]:
bdb_uniprot_level = bdb_final[bdb_final["query_type"] == "uniprot"].copy()
bdb_pdb_level = bdb_final[bdb_final["query_type"] == "pdb"].copy()

In [34]:
# Dump to file.

filtered_df.to_csv(DATA / f"{readout}_chembl.csv")